# Custom LPRnet-keras Training (Philippine Context)

This notebook is designed to train a custom LPRNet model using the Keras-based synthetic plate generation approach from the `LPRnet-keras` repository.

### Prerequisites:
1. You must upload your `LPRnet-keras` folder to your Google Drive (e.g., inside `/content/drive/MyDrive/LPRnet-keras`).
2. The folder must contain the `/fonts/` directory with `.ttf` files to generate synthetic plates.

We will override the data generator natively in this notebook to specifically generate **Philippine Plate Formats** (3 letters + 3 or 4 digits) and bypass the need for a real validation dataset.

In [ ]:
!pip install wandb

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# CHANGE THIS PATH if you uploaded LPRnet-keras to a different location in your Drive
import os
os.chdir('/content/drive/MyDrive/LPRnet-keras')
print("Current Directory:", os.getcwd())

In [ ]:
import random
import cv2
import numpy as np
from PIL import Image, ImageDraw, ImageFont
from data_aug_keras import data_augmentation
import tensorflow as tf
import tensorflow.keras as keras

# Philippine Plate Vocabulary Setup
IMAGE_SHAPE = [94, 24]
CHARS = "ABCDEFGHIJKLMNPQRSTUVWXYZ0123456789"
CHARS_DICT = {char: i for i, char in enumerate(CHARS)}
NUM_CLASS = len(CHARS) + 1  # +1 for CTC blank token

class CustomImageGenerator:
    def __init__(self, ttf_dir='./fonts/', char_set=CHARS, char_height=36):
        self.chars = char_set
        self.letters = [c for c in char_set if c.isalpha()]
        self.digits = [c for c in char_set if c.isdigit()]

        self.char_height = char_height
        self.ttf_dir = ttf_dir
        self.fonts, self.font_char_ims = self.load_fonts(ttf_dir)
        
        # Colors
        green = [0, 1, 0]
        white = [1, 1, 1]
        yellow = [0, 1, 1]
        blue = [1, 0, 0]
        self.black_text_colors = [white, yellow, green]
        self.white_text_colors = [blue]

    def random_text_plate_colors(self, min_diff=0.3, black_text=True):
        high = random.uniform(min_diff, 1.0)
        low = random.uniform(0.0, high - min_diff)
        text_color, plate_color = (low, high) if black_text else (high, low)
        return text_color, plate_color

    def load_fonts(self, folder_path):
        font_char_ims = {}
        fonts = [f for f in os.listdir(folder_path) if f.lower().endswith('.ttf')]
        for font in fonts:
            font_char_ims[font] = dict(self.generate_char_imgs(os.path.join(folder_path, font), self.char_height))
        return fonts, font_char_ims

    def generate_char_imgs(self, font_path, output_height):
        font_size = output_height * 4
        font = ImageFont.truetype(font_path, font_size)
        # getbbox returns (left, top, right, bottom), we want height
        height = max(font.getbbox(c)[3] - font.getbbox(c)[1] for c in self.chars)

        for c in self.chars:
            width = font.getbbox(c)[2] - font.getbbox(c)[0]
            im = Image.new("RGBA", (width, height), (0, 0, 0))
            draw = ImageDraw.Draw(im)
            draw.text((0, 0), c, (255, 255, 255), font=font)
            scale = float(output_height) / height
            im = im.resize((int(width * scale), output_height), Image.Resampling.LANCZOS)
            yield c, np.array(im)[:, :, 0]

    def generate_code_trial(self):
        # CUSTOM PHILIPPINE FORMAT: 3 Letters + 3 or 4 Digits
        pre_letters = [random.choice(self.letters) for _ in range(3)]
        digit_n = random.choice([3, 4])
        digits = [random.choice(self.digits) for _ in range(digit_n)]
        return ''.join(pre_letters) + '-' + ''.join(digits)

    def getOneRandomFont(self):
        return random.choice(self.fonts)

    def getCharGivenLabelFont(self, label, font):
        return self.font_char_ims[font][label], label

    def generate_images(self, number):
        images, labels = [], []
        for _ in range(number):
            char_height = self.char_height
            code = self.generate_code_trial()

            space = round(char_height * random.uniform(0.0, 0.3))
            char_spacing = [space if c != '-' else space + int(char_height * 0.2) for c in code]
            code = code.replace('-', '')

            char_ims = []
            char_font = self.getOneRandomFont()
            for c in code:
                char, _ = self.getCharGivenLabelFont(c, char_font)
                char_ims.append(char)

            char_width_sum = sum(im.shape[1] for im in char_ims)
            top_padding = round(random.uniform(0.1, 1.0) * char_height)
            bot_padding = round(random.uniform(0.1, 1.0) * char_height)
            left_padding = round(random.uniform(0.1, 1.0) * char_height)
            right_padding = round(random.uniform(0.1, 1.0) * char_height)

            Plate_h = char_height + top_padding + bot_padding
            Plate_w = char_width_sum + left_padding + right_padding + sum(char_spacing[:-1])

            text_mask = np.zeros((Plate_h, Plate_w))
            x, y = left_padding, top_padding
            for ind, char_im in enumerate(char_ims):
                ix, iy = int(x), int(y)
                text_mask[iy:iy + char_im.shape[0], ix:ix + char_im.shape[1]] = char_im
                x += char_im.shape[1] + char_spacing[ind]

            is_black_text = random.choice([True, False])
            text_color, plate_color = self.random_text_plate_colors(black_text=is_black_text)
            plate_mask = 255. - text_mask

            color = np.array(random.choice(self.black_text_colors if is_black_text else self.white_text_colors))
            w_color = color * plate_color

            Plate = np.ones((Plate_h, Plate_w, 3))
            for i in range(3):
                Plate[:, :, i] = text_mask * text_color + plate_mask * w_color[i]

            Plate = Plate.astype(np.float32)
            Plate = data_augmentation(Plate)
            images.append(cv2.resize(Plate, (94, 24)) / 256.0)
            labels.append(code)

        return images, labels


In [ ]:
gen = CustomImageGenerator()

class DataGenerator(tf.keras.utils.Sequence):
    def __init__(self, batch_size=64, steps_per_epoch=50):
        self.batch_size = batch_size
        self.steps_per_epoch = steps_per_epoch

    def __len__(self):
        return self.steps_per_epoch

    def __getitem__(self, index):
        data, labels = gen.generate_images(self.batch_size)
        gen_labels = [[CHARS_DICT[i] for i in label] for label in labels]
        
        training_set = np.array(data, dtype=np.float32)
        training_labels = np.array(gen_labels, dtype=object) # handle variable lengths
        ragged = tf.ragged.constant(training_labels).to_tensor()
        
        return training_set, ragged

# We use the same generator for training and validation
train_data_gen = DataGenerator(batch_size=64, steps_per_epoch=50)
val_data_gen = DataGenerator(batch_size=64, steps_per_epoch=10)

In [ ]:
# Load LPRNet Architecture from the cloned repository
from LPRnet.LPRnet_separable import LPRnet, CTCLoss, global_context

print("Building LPRnet from scratch...")
model = LPRnet()
model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3), loss=CTCLoss)
model.build((1, 24, 94, 3))
model.summary()

In [ ]:
import wandb
from wandb.keras import WandbCallback

# Login to Weights & Biases (Optional, you can comment this out and remove WandbCallback if not using W&B)
wandb.init(project="LPRnet-keras-PH", name="ph_custom_model")

MODEL_NAME = "lprnet_ph_custom"
MODEL_PATH = "trained_models"
os.makedirs(MODEL_PATH, exist_ok=True)

check = tf.keras.callbacks.ModelCheckpoint(
    os.path.join(MODEL_PATH, MODEL_NAME),
    monitor="val_loss",
    verbose=1,
    save_best_only=True,
    save_weights_only=True,
    mode="auto",
)

EPOCHS = 1000
print(f"Training model for {EPOCHS} epochs...")

model.fit(
    train_data_gen, 
    validation_data=val_data_gen, 
    epochs=EPOCHS,
    callbacks=[WandbCallback(), check]
)

In [ ]:
# Convert and Export to TensorFlow Lite
TFLITE_PATH = "tflite_models"
os.makedirs(TFLITE_PATH, exist_ok=True)

converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

final_tflite_path = f"{TFLITE_PATH}/{MODEL_NAME}.tflite"
with open(final_tflite_path, 'wb') as f:
    f.write(tflite_model)
    
print(f"✅ Successfully exported to {final_tflite_path}!")
print("You can now download this .tflite file from your Drive and use it in your local desktop app.")